# TECTON PNUD — entrenamiento y validación en Colab
Motor embebido y verificado por SHA256; llama al mismo código del repositorio, no reimplementa el modelo. CPU, Python 3.10–3.13.

**Uso:** Entorno de ejecución → Ejecutar todo. Para otro experimento, cambiar `CONFIG_NAME` y ejecutar desde **Configuración**; los datos ya cargados se reutilizan.

Reglas del reto: no compartir los datos fuera del equipo, no editar predicciones a mano, entregar solo por el formulario oficial antes del cierre.


In [ ]:
from pathlib import Path

# Configuración: None = la embebida (recomendada). Otras: 'first_submission', 'baseline', 'baseline-cal', 'catboost', 'catboost-cal'.
CONFIG_NAME = None
# Hilos de CPU; None conserva los de la configuración (2, como Colab gratuito).
THREADS = None
# Enlace de Drive a hackathon_datos.zip del PNUD. No publicar esta copia del notebook.
URL_DATOS = ''
# True guarda runs/ y state/ en el Drive del equipo para sobrevivir desconexiones.
USE_DRIVE = False
DRIVE_FOLDER = 'TECTON_PNUD'


In [ ]:
import base64, hashlib, io, json, shutil, sys, zipfile

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if USE_DRIVE and IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER / 'tecton_colab'
else:
    ROOT = Path.cwd() / 'tecton_colab'
ROOT.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'aee96196b370b7e6451eae909186f7b4597eb73bd2aa7655dad47161b63ce31f'
# Reemplazar el código de una versión anterior; runs/, state/ y data/ se conservan.
for folder in ['src', 'configs', 'docs']:
    shutil.rmtree(ROOT / folder, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        p = Path(item.filename)
        assert not p.is_absolute() and '..' not in p.parts
    z.extractall(ROOT)
print('Proyecto:', ROOT)


In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess

# Usar las bibliotecas que ya trae Colab; instalar solo lo que falte. Sin reinicios de sesión.
pins = {}
for line in (ROOT / 'requirements-colab.txt').read_text().splitlines():
    if '==' in line and not line.startswith('#'):
        name, version = line.split(';')[0].strip().split('==')
        pins[name.lower()] = version
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'scikit-learn': 'sklearn', 'joblib': 'joblib', 'threadpoolctl': 'threadpoolctl', 'catboost': 'catboost', 'gdown': 'gdown'}
missing = [name for name, module in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    spec = [f'{n}=={pins[n]}' if n in pins else n for n in missing]
    print('Instalando:', spec)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *spec])
for module in [m for m in sys.modules if m == 'tecton' or m.startswith('tecton.')]:
    del sys.modules[module]
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Versiones en uso:', {n: metadata.version(n) for n in ['numpy', 'pandas', 'scikit-learn', 'catboost']})


## Datos
Descarga directa de `hackathon_datos.zip` desde `URL_DATOS`; los CSV se copian sin modificar a `data/raw`. Solo si la descarga falla se pide subir el ZIP. No se imprimen filas.


In [ ]:
import re
import urllib.request

NAMES = ['entrenamiento.csv', 'prueba_equipos.csv', 'prueba_oculta.csv']
raw = ROOT / 'data/raw'
raw.mkdir(parents=True, exist_ok=True)
DATA_ZIP = ROOT / 'hackathon_datos.zip'


def descargar(url, destino):
    match = re.search(r'/d/([A-Za-z0-9_-]+)', url) or re.search(r'id=([A-Za-z0-9_-]+)', url)
    direct = 'https://drive.google.com/uc?export=download&id=' + match.group(1) if match else url
    try:
        import gdown
        gdown.download(direct, str(destino), quiet=True)
    except Exception as error:
        print('gdown falló, intento descarga directa:', type(error).__name__)
    if not destino.exists() or not zipfile.is_zipfile(destino):
        urllib.request.urlretrieve(direct, destino)
    if not zipfile.is_zipfile(destino):
        destino.unlink(missing_ok=True)
        raise ValueError('La descarga no devolvió un ZIP; revisar permisos del enlace.')


if any(not (raw / name).exists() for name in NAMES):
    if not DATA_ZIP.exists() and URL_DATOS:
        try:
            descargar(URL_DATOS, DATA_ZIP)
        except Exception as error:
            print('No se pudo descargar:', error)
    if not DATA_ZIP.exists() and Path('hackathon_datos.zip').exists():
        shutil.copy2('hackathon_datos.zip', DATA_ZIP)
    if not DATA_ZIP.exists() and IN_COLAB:
        print('Subir hackathon_datos.zip')
        uploaded = files.upload()
        name = next((n for n in uploaded if n.endswith('.zip')), None)
        if name is None:
            raise ValueError('Se esperaba hackathon_datos.zip.')
        DATA_ZIP.write_bytes(uploaded[name])
    if not DATA_ZIP.exists():
        raise FileNotFoundError('Falta hackathon_datos.zip: definir URL_DATOS o dejar el ZIP junto al notebook.')
    with zipfile.ZipFile(DATA_ZIP) as z:
        for item in z.infolist():
            if Path(item.filename).name in NAMES + ['diccionario_datos.csv', 'formato_entrega.csv'] and not item.is_dir():
                (raw / Path(item.filename).name).write_bytes(z.read(item))
faltan = [name for name in NAMES if not (raw / name).exists()]
if faltan:
    raise FileNotFoundError(f'El ZIP no trae {faltan}.')
print('CSV oficiales listos en', raw)


## Configuración
Auditoría estricta del contrato. Solo resúmenes agregados.


In [ ]:
from tecton.pipeline import dump_json
from tecton.schema import load_inputs

EMBEDDED_CONFIG = json.loads('{"name": "baseline-calibrado", "data_dir": "data/raw", "strict": true, "seed": 42, "threads": 2, "model": {"family": "hist", "iterations": 150, "learning_rate": 0.06, "depth": 6}, "features": {"use_history": true, "smoothing": 20.0}, "stress_test": true, "folds": [{"train_end": "2020-03-01", "valid_start": "2020-04-01", "valid_end": "2020-09-01"}, {"train_end": "2020-09-01", "valid_start": "2020-10-01", "valid_end": "2021-03-01"}, {"train_end": "2021-03-01", "valid_start": "2021-04-01", "valid_end": "2021-09-01"}, {"train_end": "2021-09-01", "valid_start": "2021-10-01", "valid_end": "2022-03-01"}, {"train_end": "2022-03-01", "valid_start": "2022-04-01", "valid_end": "2022-09-01"}], "calibration": {"months": 12}}')
config = json.loads((ROOT / 'configs' / f'{CONFIG_NAME}.json').read_text()) if CONFIG_NAME else EMBEDDED_CONFIG
config = {**config, 'data_dir': 'data/raw', 'strict': True}
if THREADS:
    config = {**config, 'threads': int(THREADS)}
_, audit, _, _ = load_inputs(ROOT / config['data_dir'], True)
dump_json(ROOT / 'configs/colab_run.json', config)
print(json.dumps({'config': config['name'], 'threads': config['threads'], 'audit': audit}, indent=2, ensure_ascii=False))


## Entrenamiento
Imprime AUC, RMSE-log, cobertura y segundos por fold. Cada run queda en una carpeta nueva; un fallo no reemplaza el champion.


In [ ]:
from tecton.pipeline import run

run_id = run(ROOT, ROOT / 'configs/colab_run.json')
print('Experimento:', run_id)


In [ ]:
import pandas as pd
from IPython.display import HTML, display
from tecton.pipeline import compare_runs

pointer = ROOT / 'state' / 'champion.json'
print('Champion:', json.loads(pointer.read_text())['run_id'] if pointer.exists() else 'sin champion')
columns = ['run_id', 'puntos_75_mean', 'auc_mean', 'rmse_log_mean', 'winkler_log_mean', 'coverage_80_mean', 'last_fold_auc', 'stress_auc', 'folds', 'elapsed_seconds']
table = pd.DataFrame(compare_runs(ROOT))
display(table[[c for c in columns if c in table.columns]])


## Promoción
Escribir el `run_id` y ejecutar. El harness rechaza runs incompletos, sin cinco folds y stress, o que no superan al champion bajo el mismo protocolo.


In [ ]:
PROMOTE_RUN_ID = None
PROMOTE_REASON = 'Cinco folds, stress de 12 meses y QA revisados'
if PROMOTE_RUN_ID:
    from tecton.pipeline import promote
    print(json.dumps(promote(ROOT, PROMOTE_RUN_ID, PROMOTE_REASON), indent=2, ensure_ascii=False))


In [ ]:
display(HTML((ROOT / 'runs' / run_id / 'dashboard.html').read_text()))


## Descargas
1. **Resumen para revisión**: métricas, manifiestos, auditoría y comparación; sin filas, OOF ni predicciones.
2. **Entrega**: `predicciones.csv` para el formulario, replicabilidad y declaraciones.
3. **Run para el geovisor**: el run sin `oof.csv`, para `tecton import-run` en la PC.


In [ ]:
from tecton.artifacts import review_pack

review = review_pack(ROOT)
print('Resumen para revisión:', review)
if IN_COLAB:
    files.download(str(review))


In [ ]:
EXPORT_RUN_ID = None  # None = champion si existe; si no, el último run de esta sesión.
from tecton.artifacts import bundle, run_pack, zip_folder

selected = EXPORT_RUN_ID or (json.loads(pointer.read_text())['run_id'] if pointer.exists() else globals().get('run_id'))
if not selected:
    raise ValueError('Indicar EXPORT_RUN_ID.')
delivery = ROOT / 'delivery' / selected
if not delivery.exists():
    bundle(ROOT, run_id=selected)
csv_entrega = ROOT / 'export' / f'predicciones-{selected}.csv'
csv_entrega.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(delivery / 'predicciones.csv', csv_entrega)
delivery_zip = zip_folder(delivery, ROOT / 'export' / f'entrega-{selected}.zip')
run_zip = run_pack(ROOT, selected)
print('CSV para el formulario:', csv_entrega)
print('Entrega completa:', delivery_zip)
print('Run para geovisor:', run_zip)
if IN_COLAB:
    files.download(str(csv_entrega))
    files.download(str(delivery_zip))
    files.download(str(run_zip))
